### This notebook shows how open AI works with LLM chain, how prompt templates work with LLM chain, and how chains work in LLM chain.

In [25]:
#Langchain
from langchain.tools import Tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.chat_models import ChatOpenAI
from langchain_classic.agents.agent_types import AgentType
from langchain_classic.agents import initialize_agent

In [27]:
import requests
from bs4 import BeautifulSoup

In [29]:
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables (e.g., OpenAI API keys) from a .env file located on the local system
env_path = r'F:\Chrome_Extension\vigneshwara_chrome_extension_3\backend\.env'
load_dotenv(env_path)

True

In [31]:
import warnings
warnings.filterwarnings('ignore')

### prompt templates

In [35]:
from langchain_core.prompts import PromptTemplate

# Define a simple prompt template
prompt_template = PromptTemplate(
    input_variables=["name"],
    template="Hello, {name}! How can I help you today?"
)

# Generate the prompt by filling in the variable
formatted_prompt = prompt_template.format(name="David")
print(formatted_prompt)  # Output: Hello, John! How can I help you today?


Hello, David! How can I help you today?


### Working with chat openai

In [37]:
from langchain.chat_models import ChatOpenAI

# Initialize the OpenAI Chat model
chat_model = ChatOpenAI(model="gpt-3.5-turbo")

# Send a message to the model and get the response
response = chat_model.invoke("What is Capital of USA?")
print(response)  # Output: The capital of the United States of America is Washington, D.C

content='The capital of the USA is Washington, D.C.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 13, 'total_tokens': 24, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019e1348-1b29-7b71-9e6b-a977bd3d3860-0' tool_calls=[] invalid_tool_calls=[]


### LLM Chains

In [42]:
llm = ChatOpenAI(model_name='gpt-3.5-turbo-1106')

learn_template = """
I want you to act as a consultant for a AI training
Return a list of topics and why it is important to learn in given area of AI
The description should be relevant to recent advancement in AI
What are some good topics to learn in {AI_topic}
"""

prompt_template = PromptTemplate(
    input_variables=["AI_topic"],
    template=learn_template,
)

description = "Please suggest me topics to learn in Deep learning"

prompt_template.format(AI_topic=description)

chain = prompt_template | llm

print(chain.invoke(description))

content='1. Neural Networks: Understanding the architecture and working of neural networks is crucial for deep learning as it forms the basis of many advanced deep learning models such as CNNs and RNNs. Recent advancements in neural network architectures like transformers and GANs have revolutionized various AI applications, making it an essential topic to learn.\n\n2. Convolutional Neural Networks (CNNs): CNNs are widely used in image recognition, object detection, and natural language processing tasks. Recent advancements in CNNs include the development of efficient architectures like EfficientNet and advancements in techniques for improving model performance, such as self-attention mechanisms and multi-scale feature fusion.\n\n3. Recurrent Neural Networks (RNNs) and Long Short-Term Memory (LSTM) networks: RNNs and LSTMs are essential for sequential data analysis and have seen recent advancements in the form of attention mechanisms and transformer-based architectures for better handl

### Agent

In [46]:
!pip install -U ddgs

   ---------------------------------------- 0.0/3.9 MB ? eta -:--:--
   -------- ------------------------------- 0.8/3.9 MB 4.2 MB/s eta 0:00:01
   ---------------- ----------------------- 1.6/3.9 MB 4.4 MB/s eta 0:00:01
   --------------------------- ------------ 2.6/3.9 MB 4.6 MB/s eta 0:00:01
   ------------------------------------- -- 3.7/3.9 MB 4.4 MB/s eta 0:00:01
   ---------------------------------------- 3.9/3.9 MB 4.3 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.1.7
    Uninstalling click-8.1.7:
      Successfully uninstalled click-8.1.7


In [48]:
from langchain.tools import Tool

# Define a simple tool
def my_tool_function(query: str) -> str:
    return f"Tool response: {query}"

#creating tool from function
my_tool = Tool.from_function(func=my_tool_function, name="simple_tool", description="A simple tool")
ddg_search = DuckDuckGoSearchRun()

# Initialize the LLM and the agent
llm = ChatOpenAI(model="gpt-3.5-turbo")
tools = [ddg_search,my_tool]

# Create an agent
agent = initialize_agent(tools, llm, agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)

# Use the agent
response = agent.run("What's the weather like today in London?")
print(response)  # Output: The agent will choose to use the tool or model to respond based on context



> Entering new AgentExecutor chain...
I should search for the current weather in London.
Action: duckduckgo_search
Action Input: "current weather in London"
Observation: Current weather in London ... Current weather conditions in London (United Kingdom) : ... Current weather data received from the weather station ... London, current weather: overcast. ... Weather in London right now: overcast, air temperature +10°, feels like +8°. Today, in London, expect [b]light showers[/b] throughout the day, with a chance of dry weather later.[br]With a high probability, [b]light ... In City of London, for the rest of Thursday, wet conditions will dominate, with rain falling for much of the day. ... On Thursday, in City of London ... Currently in London, expect varied conditions with mild temperatures around 12°. ... Weather in London is 12° today.
Thought:I have the information about the current weather in London.
Final Answer: The weather in London today is overcast with light showers expected 

In [83]:
summary_prompt = PromptTemplate.from_template("Summarize the following content: {content}")
llm = ChatOpenAI(model="gpt-3.5-turbo-16k")

# Create the LLM chain
summarization_chain = summary_prompt | llm

def summarize_content(content: str) -> str:
    """
    Summarize the provided content using the LLM chain.
    """
    response = summarization_chain.invoke({"content": content})
    return response.content


# Reusable summarization tool
summarize_tool = Tool(
    name="Summarizer",
    func=summarize_content,
    description="Summarizes web page content into a concise summary."
)

In [84]:
tools = [ddg_search, summarize_tool]

agent = initialize_agent(
    tools=tools,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    llm=llm,
    verbose=True,handle_parsing_errors=True
)

In [85]:
prompt = """Please tell me how to become a good footballer"""
result = agent.invoke({"input": prompt})
print(result["output"])



> Entering new AgentExecutor chain...
you can search for tips and advice on how to become a good footballer
Action: duckduckgo_search
Action Input: "tips on how to become a good footballer"
Observation: December 3, 2025 - If so, follow the tips below. Participate in both local and regional tournaments regularly. Showcase your skills on digital platforms (such as Instagram and Facebook) to improve your visibility. You need to improve the different parts of your games, learn how to identify your shortcomings and critically think of them and how to improve, then find a football academy to join, and finally, have a strong mindset and become a reliable and consistent player once you’ve joined an academy. Yes, infact if you become a player that is good, reliable and consistent, and football scouts find you, you will get paid for this! Published March 11, 2026 February 26, 2026 - A proper diet will allow you to stay energized, focused, and will reduce the risk of injury. Some basic tips for